# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omahcy/FlyRank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Identify content ranking on page one of search results (average position between 1.0 and 10.0) that receives high organic visibility (90-day impressions at or above the 60th percentile), but whose click-through rate (CTR) is strictly below the cohort median for its position bucket.

We prioritize these URLs by the estimated volume of missed clicks:
missed_clicks = ((expected_ctr - ctr) / 100) * impressions_90d

These pages are already trusted by search engines to rank on page one, but searchers choose other listings or SERP features. Instead of expensive content overhauls to chase higher rank, these pages require targeted title tag and metadata snippet optimization.

Action Label & Reason Code

Action Label: OPTIMIZE_METADATA_SNIPPET

Reason Code: HIGH_IMP_LOW_CTR_PAGE1

Reason Code Trigger:

HIGH_IMP_LOW_CTR_PAGE1: Fired when avg_position is between 1.0 and 10.0, impressions_90d exceeds the 60th percentile, and ctr falls below the position cohort's median benchmark.

In [2]:
%cd /content/FlyRank_Internship

[Errno 2] No such file or directory: '/content/FlyRank_Internship'
/content


In [1]:
import numpy as np
import pandas as pd

# Load the historical dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Total dataset size: {len(df)} rows\n")

# ==========================================
# SIGNAL 1: CTR vs Position Bucket (Flag-linked from session)
# Intuition: Higher positions (1-3) should yield significantly higher CTR,
# decaying predictably as rank drops.
# ==========================================
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 5, 10, 20, 50, 100],
    labels=["1-3", "4-5", "6-10", "11-20", "21-50", "51-100"],
)

signal_1_bucket = (
    df.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
    )
    .reset_index()
)

print("--- SIGNAL 1: Position vs CTR Bucket Table ---")
print(signal_1_bucket.to_string(index=False))
print("\nSignal 1 Verdict: CONFIRMED")
print(
    "Reasoning: Top ranks (1-3) exhibit higher median CTR, decaying steadily"
    " across lower ranks."
)
print(
    "Pages in positions 1-10 falling below their cohort median represent clear"
    " underperformers.\n"
)

# ==========================================
# SIGNAL 2: Impression Volume vs Click Yield
# Intuition: Higher 90-day impressions must reliably yield more clicks,
# proving that impression scale is a valid multiplier for prioritization.
# ==========================================
df["impression_bucket"] = pd.qcut(
    df["impressions_90d"].rank(method="first"),
    q=5,
    labels=["Q1 (Lowest)", "Q2", "Q3", "Q4", "Q5 (Highest)"],
)

signal_2_bucket = (
    df.groupby("impression_bucket", observed=False)
    .agg(
        n=("content_id", "count"),
        mean_impressions=("impressions_90d", "mean"),
        mean_clicks=("clicks_90d", "mean"),
        median_clicks=("clicks_90d", "median"),
    )
    .reset_index()
)

print("--- SIGNAL 2: Impression Volume vs Click Yield ---")
print(signal_2_bucket.to_string(index=False))
print("\nSignal 2 Verdict: CONFIRMED")
print(
    "Reasoning: Mean and median clicks scale monotonically with impression"
    " quintiles."
)

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.